# Chapter 12 - Fine-tuning Generation Models

*Exploring a two-step approach for fine-tuning generative LLMs.*

---

This notebook is for Chapter 12 of the course, covering fine-tuning techniques for Large Language Models.

In this chapter, we will explore:
1. **Supervised Fine-Tuning (SFT)** - Teaching models to follow instructions
2. **Preference Tuning (DPO)** - Aligning models with human preferences
3. **QLoRA** - Parameter-efficient fine-tuning with quantization

---

### [OPTIONAL] - Installing Packages on Google Colab

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
# %%capture
# !pip install -q accelerate==0.31.0 peft==0.11.1 bitsandbytes==0.43.1 transformers==4.41.2 trl==0.9.4 sentencepiece==0.2.0 triton==3.1.0

# Supervised Fine-Tuning (SFT)

Supervised Fine-Tuning is the process of training a pretrained language model on instruction-following data. This teaches the model to respond to user prompts in a helpful and coherent way.

The general workflow:
1. **Load instruction data** - Formatted as user-assistant conversations
2. **Apply chat template** - Format data with special tokens
3. **Configure QLoRA** - Set up efficient fine-tuning
4. **Train** - Fine-tune the model
5. **Evaluate** - Test the model's responses

## Data Preprocessing

First, we'll load and format instruction data. We use the **UltraChat** dataset, which contains high-quality conversational data.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset


# Load a tokenizer to use its chat template
template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLLama is using"""

    # Format answers
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)

    return {"text": prompt}

# Load and format the data using the template TinyLLama is using
dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k",  split="test_sft")
      .shuffle(seed=42)
      .select(range(3_000))
)
dataset = dataset.map(format_prompt)

Let's examine a formatted example to understand the chat template:

In [ ]:
# Example of formatted prompt
print(dataset["text"][2576])

### Understanding Chat Templates

The chat template formats conversations with special tokens:
- `<|user|>` - Marks the start of user input
- `<|assistant|>` - Marks the start of assistant response
- `</s>` - End-of-sequence token

This structure helps the model understand the conversation flow and distinguish between user prompts and expected responses.

## Models - Quantization

We'll use **QLoRA** (Quantized Low-Rank Adaptation) to efficiently fine-tune the model:

1. **4-bit Quantization** - Reduces memory usage by ~75%
2. **LoRA** - Only trains small adapter matrices instead of all parameters

This allows us to fine-tune a 1.1B parameter model on a single GPU!

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Use 4-bit precision model loading
    bnb_4bit_quant_type="nf4",  # Quantization type (NormalFloat4)
    bnb_4bit_compute_dtype="float16",  # Compute dtype for efficiency
    bnb_4bit_use_double_quant=True,  # Apply nested quantization for even more compression
)

# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,  # Apply quantization
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

### Key Quantization Parameters:

- **load_in_4bit**: Loads model weights in 4-bit format (vs. 16-bit or 32-bit)
- **bnb_4bit_quant_type="nf4"**: Uses NormalFloat4, a special 4-bit data type optimized for normally distributed weights
- **bnb_4bit_use_double_quant**: Further compresses the quantization constants themselves

## Configuration

### LoRA Configuration

LoRA adds small trainable matrices to specific layers. Instead of updating all weights, we only train these adapters.

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Prepare LoRA Configuration
peft_config = LoraConfig(
    lora_alpha=32,  # LoRA Scaling (typically 2x the rank)
    lora_dropout=0.1,  # Dropout for LoRA Layers (prevents overfitting)
    r=64,  # Rank - dimensionality of the low-rank matrices
    bias="none",  # Don't train bias terms
    task_type="CAUSAL_LM",  # Task type for causal language modeling
    target_modules=  # Which layers to add LoRA adapters to
     ['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

### LoRA Parameters Explained:

- **r (rank)**: Higher rank = more parameters = better performance but more memory. Common values: 8, 16, 32, 64
- **lora_alpha**: Scaling factor for the LoRA updates. Controls how much the adapters affect the base model
- **target_modules**: Which parts of the transformer to add LoRA to (query, key, value projections, etc.)

### Training Configuration

Now we set up the training hyperparameters:

In [ ]:
from transformers import TrainingArguments

output_dir = "./results"

# Training arguments
training_arguments = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,  # Batch size per GPU
    gradient_accumulation_steps=4,  # Accumulate gradients over 4 steps (effective batch size = 2*4 = 8)
    optim="paged_adamw_32bit",  # Optimizer optimized for memory efficiency
    learning_rate=2e-4,  # Learning rate
    lr_scheduler_type="cosine",  # Learning rate decay schedule
    num_train_epochs=1,  # Number of full passes through the data
    logging_steps=10,  # Log metrics every 10 steps
    fp16=True,  # Use mixed precision training for speed
    gradient_checkpointing=True  # Trade compute for memory
)

## Training!

Now we're ready to train using the **SFTTrainer** (Supervised Fine-Tuning Trainer):

In [ ]:
from trl import SFTTrainer

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",  # Which field contains the formatted text
    tokenizer=tokenizer,
    args=training_arguments,
    max_seq_length=512,  # Maximum sequence length
    peft_config=peft_config,  # LoRA configuration
)

# Train model
trainer.train()

# Save LoRA weights
trainer.model.save_pretrained("TinyLlama-1.1B-qlora")

### Merge Adapter

After training, we need to merge the LoRA adapters back into the base model for inference:

In [ ]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Merge LoRA and base model
merged_model = model.merge_and_unload()

### Inference

Let's test our fine-tuned model!

In [ ]:
from transformers import pipeline

# Use our predefined prompt template
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

# Run our instruction-tuned model
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

---

# Preference Tuning (DPO)

After supervised fine-tuning, we can further improve the model using **Direct Preference Optimization (DPO)**.

DPO trains the model to prefer better responses over worse ones by learning from preference pairs:
- **Chosen**: High-quality response
- **Rejected**: Lower-quality response

This helps align the model with human preferences for helpfulness, harmlessness, and honesty.

## Data Preprocessing

For DPO, we need a dataset with preference pairs (chosen vs rejected responses):

In [ ]:
from datasets import load_dataset

def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLLama is using"""

    # Format answers
    system = "<|system|>\n" + example['system'] + "</s>\n"
    prompt = "<|user|>\n" + example['input'] + "</s>\n<|assistant|>\n"
    chosen = example['chosen'] + "</s>\n"
    rejected = example['rejected'] + "</s>\n"

    return {
        "prompt": system + prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

# Apply formatting to the dataset and select relatively short answers
dpo_dataset = load_dataset("argilla/distilabel-intel-orca-dpo-pairs", split="train")
dpo_dataset = dpo_dataset.filter(
    lambda r:
        r["status"] != "tie" and
        r["chosen_score"] >= 8 and
        not r["in_gsm8k_train"]
)
dpo_dataset = dpo_dataset.map(format_prompt, remove_columns=dpo_dataset.column_names)
dpo_dataset

## Models - Quantization

We load the SFT model and apply quantization again for memory efficiency:

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import BitsAndBytesConfig, AutoTokenizer

# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Use 4-bit precision model loading
    bnb_4bit_quant_type="nf4",  # Quantization type
    bnb_4bit_compute_dtype="float16",  # Compute dtype
    bnb_4bit_use_double_quant=True,  # Apply nested quantization
)

# Merge LoRA and base model
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
    quantization_config=bnb_config,
)
merged_model = model.merge_and_unload()

# Load LLaMA tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=False)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

## Configuration

We'll use similar LoRA configuration as before:

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Prepare LoRA Configuration
peft_config = LoraConfig(
    lora_alpha=32,  # LoRA Scaling
    lora_dropout=0.1,  # Dropout for LoRA Layers
    r=64,  # Rank
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=  # Layers to target
     ['k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj', 'o_proj', 'down_proj']
)

# prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

DPO-specific training configuration:

In [ ]:
from trl import DPOConfig

output_dir = "./results"

# Training arguments for DPO
training_arguments = DPOConfig(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=1e-5,  # Lower learning rate for DPO
    lr_scheduler_type="cosine",
    max_steps=200,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,
    warmup_ratio=0.1
)

## Training with DPO

Now we train using the **DPOTrainer**:

In [ ]:
from trl import DPOTrainer

# Create DPO trainer
dpo_trainer = DPOTrainer(
    model,
    args=training_arguments,
    train_dataset=dpo_dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
    beta=0.1,  # Temperature parameter for DPO
    max_prompt_length=512,
    max_length=512,
)

# Fine-tune model with DPO
dpo_trainer.train()

# Save adapter
dpo_trainer.model.save_pretrained("TinyLlama-1.1B-dpo-qlora")

### Merge Both Adapters

Finally, we merge both the SFT and DPO adapters:

In [ ]:
from peft import PeftModel

# Merge LoRA and base model
model = AutoPeftModelForCausalLM.from_pretrained(
    "TinyLlama-1.1B-qlora",
    low_cpu_mem_usage=True,
    device_map="auto",
)
sft_model = model.merge_and_unload()

# Merge DPO LoRA and SFT model
dpo_model = PeftModel.from_pretrained(
    sft_model,
    "TinyLlama-1.1B-dpo-qlora",
    device_map="auto",
)
dpo_model = dpo_model.merge_and_unload()

### Test DPO Model

Let's compare the DPO-trained model with the SFT-only model:

In [ ]:
from transformers import pipeline

# Use our predefined prompt template
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""

# Run our DPO-tuned model
pipe = pipeline(task="text-generation", model=dpo_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

---

## Summary

In this notebook, you learned:

1. **Supervised Fine-Tuning (SFT)**:
   - How to format instruction data with chat templates
   - Using QLoRA for memory-efficient fine-tuning
   - Training with SFTTrainer

2. **Direct Preference Optimization (DPO)**:
   - Training on preference pairs (chosen vs rejected)
   - Further aligning models with human preferences
   - Combining SFT and DPO for best results

3. **QLoRA Components**:
   - 4-bit quantization with NormalFloat4
   - Low-rank adapters for parameter efficiency
   - Merging adapters back into the base model

## Next Steps

Now try the practice tasks to deepen your understanding:
- **Easy Tasks**: Basic instruction tuning experiments
- **Medium Tasks**: LoRA configuration and comparison studies
- **Hard Tasks**: Advanced DPO training and evaluation